# 22 — Technique: code-switch augmentation (Strategy C)

**Source.** `model-research.md` §5, Strategy C — "fine-tune-time augmentation (transliteration +
code-switching injected into training). Cheap, high value, applies on top of A or B."

Unlike A and B it is not an alternative pipeline; it is extra training data, so it stacks on
whichever inference path wins.

**Why it is plausible here specifically.** The bake-off's `zeroshot-en` regime collapsed —
sentiment fell from ~0.48 to 0.085 — which says cross-script transfer does not happen for free and
the model is largely learning each script separately. Augmentation that mixes scripts inside a
single ticket is a direct attack on that: it forces shared representations rather than five
parallel ones.

**Three augmentation operators**, all applied to training rows only, never to dev or test:

| operator | what it does |
|---|---|
| `loanword-swap` | replaces a Sinhala-script loanword with its romanized form from `singlish_overrides.py` (`කාඩ්` ↔ `card`) — the code-switching Sri Lankans actually type |
| `script-mix` | splices the first half of a ticket from one language copy and the second half from another, using the shared `id` |
| `char-noise` | random character deletion/duplication at a low rate, imitating the typo distribution real romanized text carries and ours does not |

The third exists because of the finding in
[`08_word_tokenizer_comparison.ipynb`](08_word_tokenizer_comparison.ipynb) §6: our Singlish has
**1.18% dev OOV, identical to English**, because it is rule-generated. Real romanized text is far
noisier. `char-noise` is the cheapest way to stop the model depending on an orthographic
consistency that production will not have.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, data, imbalance, metrics, models, splits, tokenize as sbtok

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
LANGS = config.LANGUAGES
POS = config.SENTIMENT_POSITIVE_CLASS
print("split sha:", splits.sha())

In [ ]:
def boot_ci(y_true, y_pred, ids, n_boot=1000, seed=42):
    # 95% CI for negative_f1, resampled over ticket id.
    yt = (np.asarray(y_true) == POS); yp = (np.asarray(y_pred) == POS)
    groups = [g.to_numpy() for _, g in pd.Series(np.arange(len(yt))).groupby(np.asarray(ids))]
    rng = np.random.default_rng(seed); n = len(groups); out = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([groups[j] for j in rng.integers(0, n, n)])
        tp = (yt[idx] & yp[idx]).sum(); fp = (~yt[idx] & yp[idx]).sum(); fn = (yt[idx] & ~yp[idx]).sum()
        out[i] = 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)
    return tuple(np.percentile(out, [2.5, 97.5]))


def fit_champion(df, task="sentiment"):
    col = data.label_column(task)
    fit = imbalance.resample(df, col, "class_weight")
    clf = models.build("tfidf-svm", class_weight="balanced", C=0.5 if task == "sentiment" else 1.0)
    clf.fit(fit[config.TEXT_COLUMN], fit[col])
    return clf

## 1. The operators

In [ ]:
import sys
sys.path.insert(0, str(REPO / "datasets" / "translation"))
try:
    from singlish_overrides import OVERRIDES
    print(f"{len(OVERRIDES)} loanword overrides loaded")
except ImportError:
    OVERRIDES = {}
    print("singlish_overrides not importable -- loanword-swap will be a no-op")

rng_global = np.random.default_rng(config.RANDOM_STATE)


def loanword_swap(text, p=0.5, rng=rng_global):
    # Sinhala-script loanword -> its romanized form, the way people actually type.
    for si, rom in OVERRIDES.items():
        if si in text and rng.random() < p:
            text = text.replace(si, rom)
    return text


def char_noise(text, rate=0.03, rng=rng_global):
    # Low-rate character deletion / duplication -- imitates real typing noise.
    out = []
    for ch in text:
        r = rng.random()
        if ch.isspace() or r > rate:
            out.append(ch)
        elif r < rate / 2:
            continue                      # drop
        else:
            out.append(ch); out.append(ch)  # duplicate
    return "".join(out)


def script_mix(df, rng=rng_global):
    # Splice two language copies of the SAME ticket id -- a genuinely code-switched row.
    pairs = []
    by_id = {i: g for i, g in df.groupby("id")}
    for tid, g in by_id.items():
        if len(g) < 2:
            continue
        a, b = g.sample(2, random_state=int(rng.integers(1e9)))[config.TEXT_COLUMN].tolist()
        wa, wb = a.split(), b.split()
        if len(wa) < 2 or len(wb) < 2:
            continue
        mixed = " ".join(wa[: len(wa) // 2] + wb[len(wb) // 2 :])
        row = g.iloc[0].copy(); row[config.TEXT_COLUMN] = mixed; row["language"] = "mixed"
        pairs.append(row)
    return pd.DataFrame(pairs)


sample = "මගේ කාඩ් එක ට්‍රැක් කරන්න"
print("original      :", sample)
print("loanword-swap :", loanword_swap(sample, p=1.0))
print("char-noise    :", char_noise("mage card eka track karanna", rate=0.12))

## 2. Build the augmented training sets

Each arm adds rows to train; dev is untouched. Sizes are reported so a gain cannot be confused
with simply having more data — the `duplicate` control arm exists for exactly that reason.

In [ ]:
train = splits.get(LANGS, "train")
dev = splits.get(LANGS, "dev")

def augment(df, kind, frac=0.3, rng=None):
    rng = rng or np.random.default_rng(config.RANDOM_STATE)
    take = df.sample(frac=frac, random_state=config.RANDOM_STATE).copy()
    if kind == "loanword-swap":
        take[config.TEXT_COLUMN] = [loanword_swap(t, rng=rng) for t in take[config.TEXT_COLUMN]]
    elif kind == "char-noise":
        take[config.TEXT_COLUMN] = [char_noise(t, rng=rng) for t in take[config.TEXT_COLUMN]]
    elif kind == "script-mix":
        take = script_mix(df.sample(frac=frac, random_state=config.RANDOM_STATE), rng=rng)
    elif kind == "duplicate":
        pass                                   # control: same rows again, no transformation
    return pd.concat([df, take], ignore_index=True)


ARMS = ["none", "duplicate", "loanword-swap", "char-noise", "script-mix"]
sets = {a: (train if a == "none" else augment(train, a)) for a in ARMS}
for a, d in sets.items():
    print(f"  {a:15s} {len(d):,} rows")

## 3. Does it help?

Same estimator throughout, dev held fixed. `duplicate` is the control: any arm that fails to beat
it has bought nothing that plain repetition would not.

In [ ]:
rows = []
for arm, dfa in sets.items():
    clf = fit_champion(dfa)
    pred = clf.predict(dev[config.TEXT_COLUMN])
    sc = metrics.score(dev.sentiment, pred, "sentiment")
    lo, hi = boot_ci(dev.sentiment, pred, dev.id.to_numpy(), n_boot=500)
    rows.append({"arm": arm, "train_rows": len(dfa), "negative_f1": sc["negative_f1"],
                 "precision": sc["negative_precision"], "recall": sc["negative_recall"],
                 "accuracy": sc["accuracy"], "ci_low": lo, "ci_high": hi})
    print(f"  {arm:15s} negative_f1 {sc['negative_f1']:.4f}  [{lo:.4f}, {hi:.4f}]", flush=True)

aug = pd.DataFrame(rows)
aug["vs none"] = aug.negative_f1 - aug.loc[aug.arm == "none", "negative_f1"].iloc[0]
aug["vs duplicate"] = aug.negative_f1 - aug.loc[aug.arm == "duplicate", "negative_f1"].iloc[0]
display(aug)
aug.to_csv(REPO / "ml" / "reports" / "technique_augmentation.csv", index=False)

In [ ]:
# Per-language -- augmentation should help the romanized tracks most if it works at all.
rows = []
for arm in ARMS:
    clf = fit_champion(sets[arm])
    pred = clf.predict(dev[config.TEXT_COLUMN])
    for lang in LANGS:
        m = (dev.language == lang).to_numpy()
        sc = metrics.score(dev.sentiment.to_numpy()[m], pred[m], "sentiment")
        rows.append({"arm": arm, "language": lang, "negative_f1": sc["negative_f1"]})
piv = pd.DataFrame(rows).pivot(index="language", columns="arm", values="negative_f1")
piv["best gain"] = piv.drop(columns=["none"]).max(axis=1) - piv["none"]
display(piv)

## 4. Verdict

Read every arm against **`duplicate`**, not against `none`. Dev's CI width is roughly ±0.08 on 68
Negative tickets, so only a large effect is detectable here at all.

The honest prior is that `char-noise` and `loanword-swap` help little on *this* data, for the same
reason Strategy A cannot be evaluated on it: our romanized text is already perfectly regular, so
augmentation is teaching robustness against a corruption the evaluation set does not contain. The
place these operators should pay off is on human-typed input — which means the right follow-up is
a small human-typed evaluation set, not more augmentation.